# Обучение модели ResNet18 для классификации AI vs Human Generated Images

Этот notebook содержит код для обучения модели ResNet18 на датасете Train_1 и оценки качества на Test_1.

## 1. Импорт библиотек

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 2. Подготовка данных

In [10]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
        
        if (self.root_dir / 'test_data').exists() and 'train_data/' in self.data['file_name'].iloc[0]:
            print("Fixing paths: replacing 'train_data/' with 'test_data/'")
            self.data['file_name'] = self.data['file_name'].str.replace('train_data/', 'test_data/')
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = self.root_dir / self.data.iloc[idx]['file_name']
        image = Image.open(img_path).convert('RGB')
        label = self.data.iloc[idx]['label']
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [11]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageDataset(
    csv_file='/data/shared_ml/vmoskalenko/ai-vs-human-generated-dataset-hw/Train_1/train.csv',
    root_dir='/data/shared_ml/vmoskalenko/ai-vs-human-generated-dataset-hw/Train_1',
    transform=train_transform
)

test_dataset = ImageDataset(
    csv_file='/data/shared_ml/vmoskalenko/ai-vs-human-generated-dataset-hw/Test_1/test.csv',
    root_dir='/data/shared_ml/vmoskalenko/ai-vs-human-generated-dataset-hw/Test_1',
    transform=test_transform
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

Fixing paths: replacing 'train_data/' with 'test_data/'
Train dataset size: 9993
Test dataset size: 3997


## 3. Создание модели ResNet18

In [12]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

print(f'Model architecture:')
print(model)

/home/vmoskalenko/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/vmoskalenko/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model architecture:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): R

## 4. Определение функции потерь и оптимизатора

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## 5. Функция обучения

In [14]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for images, labels in tqdm(dataloader, desc='Training'):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1

## 6. Функция валидации

In [15]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    epoch_precision = precision_score(all_labels, all_preds, average='weighted')
    epoch_recall = recall_score(all_labels, all_preds, average='weighted')
    
    return epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

## 7. Обучение модели

In [20]:
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

num_epochs = 10
train_losses = []
train_accs = []
train_f1s = []

log_dir = f'logs/train_1_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
writer = SummaryWriter(log_dir)
writer.add_text('Hyperparameters', f'Learning rate: 0.001\nBatch size: 32\nEpochs: 10\nOptimizer: Adam')

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
    scheduler.step()
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_f1s.append(train_f1)
    
    print(f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')
    
    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_acc, epoch)
    writer.add_scalar('F1/train', train_f1, epoch)

torch.save(model.state_dict(), 'model_v1.pth')
print('Model saved as model_v1.pth')



Epoch 1/10


Training: 100%|███████████████████████████| 313/313 [00:15<00:00, 20.45it/s]


Train Loss: 0.0489, Acc: 0.9817, F1: 0.9817

Epoch 2/10


Training: 100%|███████████████████████████| 313/313 [00:14<00:00, 21.08it/s]


Train Loss: 0.0463, Acc: 0.9829, F1: 0.9829

Epoch 3/10


...

Train Loss: 0.0484, Acc: 0.9834, F1: 0.9834

Epoch 9/10


Training: 100%|███████████████████████████| 313/313 [00:15<00:00, 20.86it/s]


Train Loss: 0.0474, Acc: 0.9827, F1: 0.9827

Epoch 10/10


Training: 100%|███████████████████████████| 313/313 [00:14<00:00, 21.19it/s]


Train Loss: 0.0507, Acc: 0.9819, F1: 0.9819
Model saved as model_v1.pth


## 8. Оценка модели на тестовом датасете

In [21]:
test_loss, test_acc, test_f1, test_precision, test_recall = validate(model, test_loader, criterion, device)

print('РЕЗУЛЬТАТЫ НА ТЕСТОВОМ ДАТАСЕТЕ Test_1')
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test F1 Score: {test_f1:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Recall:   {test_recall:.4f}')
writer.add_hparams(
    {'lr': 0.001, 'batch_size': 32, 'epochs': num_epochs},
    {
        'test/accuracy': test_acc,
        'test/f1_score': test_f1,
        'test/precision': test_precision,
        'test/recall': test_recall
    }
)
writer.add_scalar('Loss/test', test_loss, 0)
writer.add_scalar('Accuracy/test', test_acc, 0)
writer.add_scalar('F1/test', test_f1, 0)

writer.close()
print(f"TensorBoard logs saved to {log_dir}")
print(f"To view run: tensorboard --logdir={log_dir} --host=0.0.0.0 --port=6006")

Validation: 100%|█████████████████████████| 125/125 [00:04<00:00, 28.97it/s]


РЕЗУЛЬТАТЫ НА ТЕСТОВОМ ДАТАСЕТЕ Test_1
Test Loss:     0.0711
Test Accuracy: 0.9755
Test F1 Score: 0.9755
Test Precision: 0.9755
Test Recall:   0.9755
TensorBoard logs saved to logs/train_1_20260521_175406
To view run: tensorboard --logdir=logs/train_1_20260521_175406 --host=0.0.0.0 --port=6006


## 9. Дообучите модель на втором датасете и постройте DVC пайплайн

In [18]:
# Добавьте логику дообучения
# Добавьте PVC пайплайн

## 10. Напишите вывод о полученных результатах

...